# 🚀 Phase 0: System-Setup, Hardware-Verification & Phasen-Steuerung

Dieses Notebook orchestriert das Gesamtsystem:
1. **Umgebung & Auth:** Setzt den NGC Key und meldet Docker an der Registry an.
2. **Hardware Check:** Garantiert GPU-Treiber & Auslastung.
3. **Phasen-Steuerung:** Schaltet dynamisch zwischen **Phase 1 (Großes SDG-LLM)** und **Phase 2 (Kleines Mistral-LLM + NeMo Customizer)** um.

## 1. Umgebung prüfen & Bibliotheken importieren

In [1]:
import subprocess
import torch
import socket

print("=== 1. Hardware- & GPU-Check im Container ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# NVIDIA-SMI Integration
print("\n--- 1.1 NVIDIA-SMI GPU Status ---")
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(result.stdout)
except Exception as e:
    print(f"Konnte nvidia-smi nicht ausführen: {e}")

print("\n=== 2. Docker-Netzwerk & Container-Erreichbarkeit prüfen ===")
# Dienste und ihre internen Standard-Ports im Docker-Netzwerk
services = {
    "prometheus": 9090,
    "grafana": 3000,
    "dcgm-exporter": 9400
}

for service, port in services.items():
    try:
        # Versuche eine TCP-Verbindung zum Service im Docker-Netzwerk aufzubauen
        with socket.create_connection((service, port), timeout=2):
            print(f"[ERFOLG] Service '{service}' ist über Port {port} im Docker-Netzwerk erreichbar.")
    except (socket.timeout, socket.error):
        print(f"[WARNUNG] Konnte keine Verbindung zu '{service}' auf Port {port} herstellen.")

ModuleNotFoundError: No module named 'torch'

## 2. Environment & NGC Authentifizierung

In [9]:
import os
import subprocess

print("=== 3. NGC Authentifizierung & Konfiguration ===")

api_key = os.getenv("NGC_API_KEY")

if api_key:
    # NGC speichert den API-Key standardmäßig in dieser Konfigurationsdatei
    ngc_config_dir = os.path.expanduser("~/.ngc")
    os.makedirs(ngc_config_dir, exist_ok=True)
    
    config_file_path = os.path.join(ngc_config_dir, "config")
    
    # Konfigurationsdatei direkt schreiben (vermeidet den Aufruf der binären CLI)
    config_content = f"apikey = {api_key}\nformat = json\n"
    with open(config_file_path, "w") as f:
        f.write(config_content)
        
    print("[ERFOLG] NGC API-Key wurde direkt in die Konfiguration geschrieben.")
else:
    print("[INFO] Kein 'NGC_API_KEY' gefunden. (Für lokale Modelle wie dein .nemo-File ist das unkritisch).")
    print("[TIPP] Setze den Key im Notebook mit: os.environ['NGC_API_KEY'] = 'dein_api_key'")

print("\n=== 4. Data Curation Verzeichnisse vorbereiten ===")
raw_data_dir = "/data/nemo-fraud-detection/data/raw"
processed_data_dir = "/data/nemo-fraud-detection/data/sft"

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

print(f"[INFO] Rohdaten-Ordner bereit: {raw_data_dir}")
print(f"[INFO] Verarbeiteter Ordner bereit: {processed_data_dir}")
print("[ERFOLG] Setup-Vorbereitung erfolgreich abgeschlossen!")

=== 3. NGC Authentifizierung & Konfiguration ===
[ERFOLG] NGC API-Key wurde direkt in die Konfiguration geschrieben.

=== 4. Data Curation Verzeichnisse vorbereiten ===
[INFO] Rohdaten-Ordner bereit: /data/nemo-fraud-detection/data/raw
[INFO] Verarbeiteter Ordner bereit: /data/nemo-fraud-detection/data/sft
[ERFOLG] Setup-Vorbereitung erfolgreich abgeschlossen!


---
## 3. Hardware & GPU-Monitoring Check



In [11]:
import logging
import subprocess
import torch

# Logging konfigurieren, damit die Ausgaben sauber angezeigt werden
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def run_hardware_check():
    logging.info("🔍 Starte Hardware- und GPU-Prüfung...")

    if not torch.cuda.is_available():
        raise RuntimeError("❌ Keine GPU gefunden! Das System benötigt eine NVIDIA L40S.")

    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    logging.info(f"✔ GPU erkannt: {gpu_name}")
    logging.info(f"✔ Verfügbarer VRAM: {vram:.2f} GB")

    # Treiber- und Monitoring-Check via nvidia-smi
    try:
        smi_result = subprocess.run(
            "nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader",
            shell=True, capture_output=True, text=True, check=True
        )
        logging.info(f"✔ Live-Metriken (nvidia-smi): {smi_result.stdout.strip()}")
    except Exception as e:
        logging.warning(f"⚠️ Konnte nvidia-smi Metriken nicht abrufen: {e}")

# Ausführung Hardware Check
run_hardware_check()
logging.info("🎉 System-Setup & Verifizierung komplett abgeschlossen!")

2026-08-22 12:42:29,606 [INFO] 🔍 Starte Hardware- und GPU-Prüfung...
2026-08-22 12:42:29,607 [INFO] ✔ GPU erkannt: NVIDIA L40S
2026-08-22 12:42:29,608 [INFO] ✔ Verfügbarer VRAM: 44.39 GB
2026-08-22 12:42:29,664 [INFO] ✔ Live-Metriken (nvidia-smi): 0 %, 3 MiB
2026-08-22 12:42:29,664 [INFO] 🎉 System-Setup & Verifizierung komplett abgeschlossen!


---
## 4. Prometheus und Grafana Setup

Ersetze die IP Adressen mit der öffentlichen IP Adresse, von dem aus Prometheus und Grafana erreichbar sind.

In [13]:
import logging
import requests

# Konfiguration des Loggings
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def check_monitoring_stack():
    logging.info("=== 4. Monitoring-Setup Check (Prometheus & Grafana) ===")
    
    monitoring_services = {
        "Prometheus": "http://18.153.96.36:9090/-/healthy",
        "Grafana": "http://18.153.96.36:3000/api/health"
    }
    
    for name, url in monitoring_services.items():
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                logging.info(f"✅ {name} ist bereit und antwortet (Status 200).")
            else:
                logging.warning(f"⚠️ {name} antwortet mit Statuscode: {response.status_code}")
        except Exception as e:
            logging.error(f"❌ {name} nicht erreichbar unter {url}. Fehler: {e}")

    # Prüfung, ob DCGM-Exporter (der die GPU-Daten an Prometheus liefert) läuft
    try:
        dcgm_metrics = requests.get("http://dcgm-exporter:9400/metrics", timeout=5)
        if "DCGM_FI_DEV_GPU_UTIL" in dcgm_metrics.text:
            logging.info("✅ DCGM-Exporter liefert GPU-Metriken!")
        else:
            logging.warning("⚠️ DCGM-Exporter läuft, aber liefert keine GPU-Metriken.")
    except Exception:
        logging.error("❌ DCGM-Exporter auf Port 9400 nicht erreichbar.")

check_monitoring_stack()

2026-08-22 12:46:20,539 [INFO] === 5. Monitoring-Setup Check (Prometheus & Grafana) ===
2026-08-22 12:46:20,543 [INFO] ✅ Prometheus ist bereit und antwortet (Status 200).
2026-08-22 12:46:20,546 [INFO] ✅ Grafana ist bereit und antwortet (Status 200).
2026-08-22 12:46:20,549 [INFO] ✅ DCGM-Exporter liefert GPU-Metriken!


In [14]:
import os
os.environ["WANDB_API_KEY"] = "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"

In [ ]:
import logging
import os
import wandb

# Logging konfigurieren
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def setup_weights_and_biases():
    logging.info("=== 6. Weights & Biases (W&B) Setup & Check ===")
    
    # Prüfen, ob der API-Key in den Umgebungsvariablen liegt
    wandb_api_key = os.getenv("WANDB_API_KEY")
    
    if wandb_api_key:
        logging.info("✔ WANDB_API_KEY gefunden. Starte Authentifizierung...")
        try:
            # Versuche den Login durchzuführen
            wandb.login(key=wandb_api_key)
            logging.info("✅ Weights & Biases erfolgreich authentifiziert!")
            
            # Kurzer Test-Run, um die Verbindung zu verifizieren
            run = wandb.init(project="nemo-fraud-detection-setup", reinit=True, mode="online")
            logging.info(f"✅ W&B Test-Run gestartet (ID: {run.id})")
            wandb.finish()
            logging.info("✅ W&B Test-Run erfolgreich beendet.")
            
        except Exception as e:
            logging.error(f"❌ Fehler bei der W&B-Authentifizierung: {e}")
    else:
        logging.warning("⚠️ Kein 'WANDB_API_KEY' gefunden.")
        logging.info("💡 Tipp: Setze den Key im Notebook mit: os.environ['WANDB_API_KEY'] = 'dein_key'")
        logging.info("ℹ️ Das Training funktioniert auch ohne W&B (wird dann übersprungen oder nur lokal geloggt), aber Grafiken fehlen.")

# Ausführung
setup_weights_and_biases()

In [18]:
import requests

try:
    response = requests.get("http://172.17.0.1:8800/v1/models", timeout=3)
    if response.status_code == 200:
        print("✅ NIM-Server ist erreichbar!")
except Exception:
    print("❌ NIM-Server antwortet noch nicht.")

✅ NIM-Server ist erreichbar!


In [1]:
import os
import urllib.request
import json
from pathlib import Path
from dotenv import load_dotenv

def check_nemo_environment():
    print("=== 1. .env & Umgebungsvariablen laden ===")
    # Versuche .env zu laden (falls im aktuellen Verzeichnis oder unter /data vorhanden)
    load_dotenv()
    
    hf_token = os.getenv("HF_TOKEN")
    ngc_key = os.getenv("NGC_API_KEY")
    
    print(f"🔑 HF_TOKEN vorhanden: {'✅ Ja' if hf_token else '❌ Nein'}")
    print(f"🔑 NGC_API_KEY vorhanden: {'✅ Ja' if ngc_key else '❌ Nein'}")
    
    print("\n=== 2. Verzeichnis-Check (/data) ===")
    data_path = Path("/data")
    if data_path.exists():
        print(f"✅ Mount-Verzeichnis '/data' ist erreichbar.")
        # Zeige kurz den Inhalt von /data an
        contents = [p.name for p in data_path.iterdir()]
        print(f"📁 Inhalt von /data: {contents}")
    else:
        print("⚠️ Warnung: Das Verzeichnis '/data' wurde nicht gefunden oder ist nicht gemountet.")
        
    print("\n=== 3. Verbindung zum LLM-Container (nim-llama3) testen ===")
    # Innerhalb des Docker-Netzwerks sprechen wir den Container über seinen Namen an:
    url = "http://nim-llama3:8000/v1/models"
    
    try:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            models = [m.get('id') for m in data.get('data', [])]
            print(f"🚀 Verbindung erfolgreich! Erreichbare Modelle auf dem NIM-Server: {models}")
    except Exception as e:
        print(f"ℹ️ Konnte den NIM-Server (nim-llama3) noch nicht erreichen: {e}")
        print("   (Hinweis: Das LLM lädt beim ersten Start möglicherweise noch im Hintergrund.)")

if __name__ == "__main__":
    check_nemo_environment()

=== 1. .env & Umgebungsvariablen laden ===
🔑 HF_TOKEN vorhanden: ✅ Ja
🔑 NGC_API_KEY vorhanden: ✅ Ja

=== 2. Verzeichnis-Check (/data) ===
✅ Mount-Verzeichnis '/data' ist erreichbar.
📁 Inhalt von /data: ['docker', 'containerd', '.docker_storage_configured', '.ipynb_checkpoints', 'evaluation', 'nim-cache', 'nemo-guardrails-offline', 'protobuf-fix', '.Trash-1000', 'monitoring', 'data', 'llama3_2_3b.nemo', 'nemo_experiments', '.Trash-0', 'nemo-fraud-detection-4', 'nemo-fraud-detection']

=== 3. Verbindung zum LLM-Container (nim-llama3) testen ===
🚀 Verbindung erfolgreich! Erreichbare Modelle auf dem NIM-Server: ['meta/llama-3.1-8b-instruct']
